# Lab 3 Assignment: Independent ReAct Review

Use this notebook after you finish `03b_lab_notebook.ipynb`. The goal here is to apply the same three-tool ReAct workflow with less guidance so you can show that you understand how one observation leads to the next tool call.

In this assignment, you will:
- review the same staged evidence package
- predict a reasonable tool order before the loop starts
- run the ReAct loop one turn at a time
- compare your manual walkthrough with `ReactAgent`
- explain where the evidence limited your conclusion

Keep `03b_lab_notebook.ipynb` nearby if you need the guided example, but make your own choices here.


In [ ]:
import csv
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

# This setup cell assumes you opened the notebook from this lab folder.
# It loads this lab's .env, adds src/ to the import path, and prepares the case data.
LAB_NAME = 'lab3_react_pattern'

lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(
        f'Open this notebook from the {LAB_NAME} folder.'
    )

repo_root = lab_dir.parent
env_example_path = lab_dir / '.env.example'
if not env_example_path.exists():
    raise FileNotFoundError(f'Expected .env.example in {LAB_NAME}.')

env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError(
        f'Expected .env in this folder. Copy .env.example to .env first.'
    )

src_dir = repo_root / 'src'
if str(src_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(src_dir.resolve()))

load_dotenv(env_path, override=True)

MODEL = os.getenv('MODEL')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL')
if not MODEL or not OLLAMA_BASE_URL:
    raise ValueError(f'MODEL or OLLAMA_BASE_URL is missing from {env_path}')

data_dir = lab_dir / 'data'
if not data_dir.exists():
    raise FileNotFoundError('Could not find the Lab 3 data folder')

from agentic_patterns.tool_pattern.tool import tool
from agentic_patterns.utils.extraction import extract_tag_content

client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

print('Repo root:', repo_root)
print('Lab folder:', lab_dir)


## Task 1

Review the assignment question and the required response format. This assignment keeps the same four-part answer structure as the guided notebook, but the question no longer forces a fixed tool order.


In [ ]:
ASSIGNMENT_QUESTION = (
    'Determine whether the Signal attachment attempt happened during the reported unattended interval '
    'and use network restoration evidence to avoid overstating confirmed delivery. '
    'Use exactly one tool call per turn. Decide the next tool based on the latest observation rather than a fixed script. '
    'Before answering, make sure you have collected evidence about the unattended interval, the Signal attachment attempt, and the network restore time. '
    'Use this exact structure: ReAct step log: a numbered list with one short line for each tool step; '
    'unattended-interval evidence: list the start and end timestamps; '
    'Signal attachment timing evidence: list the attempt timestamp and say whether it is inside the interval; '
    'connectivity context and evidence-bounded conclusion: list the network restore time and then write one careful conclusion sentence that does not claim confirmed delivery.'
)

assignment_summary = {
    'assignment_question': 'Did the Signal attachment attempt happen during the unattended interval, and what does the network evidence allow us to say about delivery?',
    'core_evidence_targets': [
        'reported unattended interval',
        'Signal attachment attempt timestamp',
        'network restoration time',
    ],
    'required_response_sections': [
        'ReAct step log',
        'unattended-interval evidence',
        'Signal attachment timing evidence',
        'connectivity context and evidence-bounded conclusion',
    ],
    'manual_goal': 'Use each observation to decide the next step instead of following a fixed order.',
}

assignment_summary


## Task 2

Review the staged case manifest again before you start calling tools. This keeps the incident window, evidence files, and case framing visible while you make your own ReAct decisions.


In [ ]:
# Load the manifest, which is the summary file for this staged evidence package.
case_manifest = json.loads((data_dir / 'artifact_manifest.json').read_text(encoding='utf-8'))

# Show the files in the data folder in a stable order so every student sees the same list.
artifact_files = sorted(path.name for path in data_dir.iterdir() if path.is_file())

# Display only the case details students need before using the tools.
{
    "case_id": case_manifest["case_id"],
    "device": case_manifest["device"],
    "incident_window_utc": case_manifest["incident_window_utc"],
    "interval_source": case_manifest["interval_source"],
    "artifact_files": artifact_files,
}


## Task 3

Load the same three forensic tools from the guided notebook. The tool set is unchanged; the main difference in this assignment is that you must decide whether each observation is enough to move forward.


In [ ]:
# Helper function: read one CSV evidence file and return its rows.
# Each row becomes a small dictionary, such as {"timestamp_utc": "..."}.
def read_csv(filename: str) -> list[dict]:
    with (data_dir / filename).open(encoding='utf-8') as handle:
        return list(csv.DictReader(handle))


# Tool 1: return the reported start and end of the unattended interval.
# The ReAct loop needs this first so later timestamps have a time window to compare against.
@tool
def get_incident_window() -> str:
    """Return the start and end timestamps for the reported unattended interval."""
    rows = read_csv('incident_window.csv')
    # Pick the row that marks when the phone became unattended.
    start = next(row for row in rows if row['event'] == 'unattended_interval_start')
    # Pick the row that marks when the phone was recovered.
    end = next(row for row in rows if row['event'] == 'unattended_interval_end')
    return json.dumps(
        {
            'start': start['timestamp_utc'],
            'end': end['timestamp_utc'],
            'start_source': start['source'],
            'end_source': end['source'],
            'start_note': start['notes'],
            'end_note': end['notes'],
        }
    )


# Tool 2: return the Signal attachment-attempt record.
# This is the event we compare against the unattended interval.
@tool
def get_message_attempt(app: str) -> str:
    """Return the first recorded messaging attachment attempt for the requested app."""
    rows = read_csv('messaging_events.csv')
    # Find the first messaging record whose app name matches the requested app.
    match = next((row for row in rows if row['app'].lower() == app.lower()), None)
    if match is None:
        return json.dumps({'error': f'No messaging event found for {app}.'})
    return json.dumps(match)


# Tool 3: return when mobile data came back online.
# This does not prove delivery; it helps keep the final claim evidence-bounded.
@tool
def get_network_restore_time() -> str:
    """Return the timestamp when mobile data connectivity was restored."""
    rows = read_csv('network_events.csv')
    # Find the network row where the status changed back to restored.
    restored = next(row for row in rows if row['status'] == 'restored')
    return restored['timestamp_utc']


# Store the tools by name so the ReAct loop can run the tool the model asks for.
available_tools = {
    'get_incident_window': get_incident_window,
    'get_message_attempt': get_message_attempt,
    'get_network_restore_time': get_network_restore_time,
}


## Task 4

Inspect the tool schemas and then predict a reasonable tool order before you run the loop. The prediction does not need to be perfect. Its purpose is to make your reasoning visible so you can compare it with the actual ReAct run later.


In [ ]:
tool_schema_preview = {
    get_incident_window.name: json.loads(get_incident_window.fn_signature),
    get_message_attempt.name: json.loads(get_message_attempt.fn_signature),
    get_network_restore_time.name: json.loads(get_network_restore_time.fn_signature),
}

tool_schema_preview


In [ ]:
student_prediction = {
    'predicted_first_tool': '',
    'why_first': '',
    'predicted_second_tool': '',
    'why_second': '',
    'predicted_third_tool': '',
    'why_third': '',
}

print(json.dumps(student_prediction, indent=2))

missing_prediction_fields = [
    key for key, value in student_prediction.items() if not str(value).strip()
]
if missing_prediction_fields:
    print(f'\nReplace the empty strings before submission: {missing_prediction_fields}')


## Task 5

Set up the independent manual ReAct run. This assignment uses the same ReAct system prompt as the guided notebook. The difference is in the user question: it asks the model to choose the next tool from the latest observation, while still gathering the same three evidence types before concluding.


In [ ]:
REACT_SYSTEM_PROMPT = """
## Role
You are a function-calling AI model that reasons and acts in a loop until you can deliver a final response to the user.

## Loop Structure
Each iteration of your loop consists of four stages:

1. Thought — reason about what you need next. Wrap in <thought></thought> tags.
2. Action — call one tool in <tool_call></tool_call> tags.
3. Observation — the tool result will be returned in <observation></observation> tags.
4. Response — when you have enough evidence, answer in <response></response> tags.

## Tool Call Format
<tool_call>
{"name": <function-name>, "arguments": <args-dict>, "id": <monotonically-increasing-id>}
</tool_call>

## Available Tools
<tools>
%s
</tools>

## Additional Constraints
- Use exactly one tool call at a time.
- Do not skip directly to a final answer when evidence is still missing.
- Keep the final answer evidence-bounded.
"""


In [ ]:
# Combine the tool descriptions into one block the model can read.
tools_signature = ',\n'.join(
    [
        get_incident_window.fn_signature,
        get_message_attempt.fn_signature,
        get_network_restore_time.fn_signature,
    ]
)
formatted_system_prompt = REACT_SYSTEM_PROMPT % tools_signature

# This is the less-scripted assignment question.
USER_QUESTION = ASSIGNMENT_QUESTION

# This list is the model's short-term memory for the assignment run.
assignment_chat_history = [
    {'role': 'system', 'content': formatted_system_prompt},
    {'role': 'user', 'content': f'<question>{USER_QUESTION}</question>'},
]


# Run one ReAct turn: model thinks, chooses a tool, receives an observation.
def run_single_react_turn(history: list[dict]):
    # Ask the model what to do next using the conversation so far.
    output = client.chat.completions.create(messages=history, model=MODEL).choices[0].message.content
    print(output)
    # Save the model's thought/action message so the next turn has context.
    history.append({'role': 'assistant', 'content': output})

    # Look inside the model output for a <tool_call>...</tool_call> block.
    tool_call = extract_tag_content(output, tag='tool_call')
    if not tool_call.found:
        # If there is no tool call, the model may already be trying to answer.
        return output, None, None

    # Convert the tool-call JSON text into a Python dictionary.
    tool_call_dict = json.loads(tool_call.content[0])
    # Use the tool name from the model to pick and run the matching local tool.
    tool_result = available_tools[tool_call_dict['name']].run(**tool_call_dict['arguments'])
    print('\nTool call:', tool_call_dict)
    print('Tool result:', tool_result)
    # Feed the tool result back as an observation for the next ReAct turn.
    history.append({'role': 'user', 'content': f'<observation>{tool_result}</observation>'})
    return output, tool_call_dict, tool_result


## Task 6

Before you run the first turn, look back at `student_prediction`. After the cell runs, compare the model's first tool call with your predicted first step.


In [ ]:
output_1, tool_call_1, tool_result_1 = run_single_react_turn(assignment_chat_history)


## Task 7

Run the second turn. Use the first observation to judge whether the next tool call makes sense. If you think a different tool would have been better, keep that note for the reflection.


In [ ]:
output_2, tool_call_2, tool_result_2 = run_single_react_turn(assignment_chat_history)


## Task 8

Run the third turn. By this point, the model should usually have enough evidence to prepare a bounded answer.


In [ ]:
output_3, tool_call_3, tool_result_3 = run_single_react_turn(assignment_chat_history)


## Task 9

Ask for the final answer after the collected observations. If the third turn already returned a final response instead of a tool call, this cell will reuse that response.


In [ ]:
if tool_call_3 is None:
    manual_final_answer = output_3
else:
    manual_final_answer = client.chat.completions.create(
        messages=assignment_chat_history,
        model=MODEL,
    ).choices[0].message.content

print(manual_final_answer)


## Task 10

Review the actual manual tool order before you compare it with `ReactAgent`.


In [ ]:
manual_tool_log = [
    tool_call['name']
    for tool_call in [tool_call_1, tool_call_2, tool_call_3]
    if tool_call is not None
]

manual_tool_log


## Task 11

Now let `ReactAgent` investigate the same assignment question. Compare its tool order and final wording with your manual walkthrough. If the agent makes a choice you disagree with, explain that in the reflection instead of silently accepting it.


In [ ]:
from agentic_patterns.react_pattern.react_agent import ReactAgent

react_agent = ReactAgent(
    tools=[get_incident_window, get_message_attempt, get_network_restore_time],
    client=client,
    model=MODEL,
)

react_agent_report = react_agent.run(user_msg=ASSIGNMENT_QUESTION)
print(react_agent_report)


## Task 12: Reflection and Submission

After you read both reports, replace the empty strings in the next cell. Use short plain-language answers.

Focus especially on whether the observation sequence supported the final conclusion and whether any claim became too strong.


In [ ]:
predicted_order = ' -> '.join(
    [
        student_prediction['predicted_first_tool'],
        student_prediction['predicted_second_tool'],
        student_prediction['predicted_third_tool'],
    ]
).strip(' ->')
manual_order = ' -> '.join(manual_tool_log)

assignment_reflection = {
    'predicted_tool_order': predicted_order,
    'manual_tool_order': manual_order,
    'did_the_manual_run_follow_a_reasonable_order': '',
    'one_observation_that_changed_the_next_step': '',
    'manual_conclusion_about_delivery': '',
    'reactagent_conclusion_about_delivery': '',
    'one_claim_you_kept_evidence_bounded': '',
    'which_report_you_trust_more_and_why': '',
}

print(json.dumps(assignment_reflection, indent=2))

missing_reflection_fields = [
    key for key, value in assignment_reflection.items() if not str(value).strip()
]
if missing_reflection_fields:
    print(f'\nComplete these fields before submission: {missing_reflection_fields}')


## Submission

Save the notebook with:
- the completed `student_prediction` from Task 4
- the three manual ReAct turns from Tasks 6 to 8
- the manual final answer from Task 9
- the `manual_tool_log` from Task 10
- the `ReactAgent` run from Task 11
- the completed `assignment_reflection` from Task 12
